# Correlation Analysis

## Overview

Correlation quantifies the linear (Pearson) or monotonic (Spearman, Kendall) relationship between two variables. Correlation is a tool for generating hypotheses, not confirming them.

| Coefficient | Measures | Assumes | Use when |
|---|---|---|---|
| **Pearson r** | Linear association | Continuous, approx. normal, no outliers | Symmetric, no heavy tails |
| **Spearman ρ** | Monotonic rank association | Ordinal or continuous | Skewed, ordinal, or with outliers |
| **Kendall τ** | Concordant pairs | Ordinal or continuous | Small n; more robust than Spearman |
| **Cramér's V** | Association between categoricals | Categorical × categorical | Any nominal variables |

**Critical caveat:** correlation does not imply causation, and zero correlation does not imply independence (only linear independence for Pearson).

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid', palette='muted')

n = 250
elevation = rng.uniform(50, 400, n)
sites = pd.DataFrame({
    'site_id':    [f'SITE_{i:03d}' for i in range(1, n+1)],
    'catchment':  rng.choice(['North', 'South', 'East', 'West'], n),
    'elevation':  elevation.round(1),
    'nitrate':    (rng.gamma(2, 2, n) + 0.008*elevation).round(2),
    'phosphorus': (rng.gamma(1.5, 1.5, n) + 0.005*elevation).round(2),
    'ph':         (rng.normal(7.5, 0.3, n) - 0.002*elevation).round(2),
    'richness':   np.clip((20 - 0.03*elevation + rng.normal(0, 4, n)).round().astype(int), 2, 40),
    'treatment':  rng.choice(['control', 'restored'], n),
})
print(sites.shape)
sites.head()

---
## Pearson, Spearman, and Kendall Correlations

In [ ]:
numeric_cols = ['elevation', 'nitrate', 'phosphorus', 'ph', 'richness']

pearson_r  = sites[numeric_cols].corr(method='pearson')
spearman_r = sites[numeric_cols].corr(method='spearman')

print('Pearson correlation matrix:')
print(pearson_r.round(3))
print('\nSpearman rank correlation matrix:')
print(spearman_r.round(3))

# Pairwise correlations with p-values and CIs
pairs = list(combinations(numeric_cols, 2))
corr_table = []
for v1, v2 in pairs:
    x, y = sites[v1].values, sites[v2].values
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    # Fisher z CI for Pearson r
    z  = np.arctanh(r_p)
    se = 1/np.sqrt(len(x)-3)
    ci_lo, ci_hi = np.tanh(z - 1.96*se), np.tanh(z + 1.96*se)
    corr_table.append({'var1':v1,'var2':v2,
                       'pearson_r':round(r_p,3),'pearson_p':round(p_p,4),
                       'ci_95':f'[{ci_lo:.3f}, {ci_hi:.3f}]',
                       'spearman_r':round(r_s,3),'spearman_p':round(p_s,4)})

print('\nPairwise correlations with 95% CIs (Pearson):')
print(pd.DataFrame(corr_table).to_string(index=False))

---
## Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

mask = np.triu(np.ones_like(pearson_r, dtype=bool))

for ax, mat, title in zip(axes,
                           [pearson_r, spearman_r],
                           ['Pearson r', 'Spearman ρ']):
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, mask=mask, ax=ax,
                square=True, linewidths=0.5)
    ax.set_title(title)

plt.suptitle('Correlation Matrices (lower triangle)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Pairplot: Scatter Matrix

In [ ]:
# Pairplot with treatment as hue — reveals group structure
pair_vars = ['nitrate', 'phosphorus', 'ph', 'richness']
g = sns.pairplot(
    sites[pair_vars + ['treatment']],
    hue='treatment',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    palette='muted'
)
g.figure.suptitle('Scatter Matrix by Treatment', y=1.01, fontsize=13)
plt.show()

---
## Non-Linear Relationships and Correlation Limits

In [ ]:
# Demonstrate: Pearson r can be near zero for strong non-linear relationships
x_nl = np.linspace(-3, 3, 300)
y_quad = x_nl**2 + rng.normal(0, 0.3, 300)   # U-shaped
y_sin  = np.sin(2*x_nl) + rng.normal(0, 0.2, 300)  # periodic

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y, label in zip(axes, [y_quad, y_sin], ['Quadratic', 'Sinusoidal']):
    r_p, _ = stats.pearsonr(x_nl, y)
    r_s, _ = stats.spearmanr(x_nl, y)
    ax.scatter(x_nl, y, alpha=0.4, s=10, color='steelblue')
    ax.set_title(f'{label}\nPearson r={r_p:.3f}  |  Spearman ρ={r_s:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('Pearson r Misses Non-Linear Associations', fontsize=13)
plt.tight_layout()
plt.show()
print('→ Always scatter-plot before trusting r = 0 as evidence of no association')

---

## Common Pitfalls

**1. Interpreting Pearson r without checking scatter plots**  
Pearson r = 0 does not mean no association — it means no *linear* association. A strong U-shaped or periodic relationship produces r near zero. Always scatter-plot variable pairs before reporting correlations.

**2. Using Pearson r on skewed variables or data with outliers**  
Pearson r is sensitive to outliers and non-normality. A single extreme observation can inflate or deflate r substantially. For skewed variables (nitrate, phosphorus), use Spearman ρ, which is based on ranks and inherently robust to outliers.

**3. Confusing statistical significance with practical significance**  
With n = 250, a correlation of r = 0.13 is statistically significant (p < 0.05) but explains only 1.7% of variance. Always report r² alongside the p-value and interpret whether the association is practically meaningful in context.

**4. Not correcting for multiple comparisons in a correlation matrix**  
Testing 10 pairwise correlations at α = 0.05 gives a 40% chance of at least one false positive. Apply Benjamini-Hochberg FDR correction to all p-values in a correlation matrix before drawing conclusions. See `03_statistical_inference/multiple_testing.ipynb`.

**5. Treating high inter-predictor correlations as redundant without investigation**  
High correlation between predictors (e.g. nitrate and phosphorus, r = 0.7) signals potential multicollinearity but does not mean one should be dropped — both may have independent biological relevance. Investigate VIF in any regression that includes both.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*